# 06 — Failure Analysis

Examines misclassified val pairs:
- Confirms no train/val leakage
- Shows FP and FN tile pairs (reference + query) with score annotations
- Plots score distributions for correct vs. incorrect predictions
- Checks for filename-level near-duplicates across the split

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/path/to/model_states/best.pt"
PROJECT_DIR = "/path/to/ct_classifier"
FIGURES_DIR = "figures"

N_SHOW  = 4    # FP/FN pairs to display
SEED    = 0

In [ ]:
import os, sys, random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.dataset import BleachDataset
from ct_classifier.model import CustomResNet
from utils.evaluate import get_model_preds, check_leakage
from utils.viz import to_display_rgb, percentile_stretch

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()

In [ ]:
# ── Leakage check + inference ─────────────────────────────────────────────────
dl_kwargs = dict(
    batch_size=cfg.get("batch_size", 32),
    shuffle=False,
    num_workers=cfg.get("num_workers", 0),
)
dl_train = DataLoader(BleachDataset(cfg, split="train", eval=True), **dl_kwargs)
dl_val   = DataLoader(BleachDataset(cfg, split="val",   eval=True), **dl_kwargs)

n_train, n_val = check_leakage(dl_train, dl_val)
print(f"Leakage check passed — train: {n_train} IDs, val: {n_val} IDs")

val_df = get_model_preds(model, dl_val, device=device)
val_df["pred"]    = (val_df["softmax_bleach_scores"] > 0.5).astype(int)
val_df["correct"] = val_df["pred"] == val_df["labels"]

fp_df = val_df[(val_df["pred"] == 1) & (val_df["labels"] == 0)]
fn_df = val_df[(val_df["pred"] == 0) & (val_df["labels"] == 1)]
print(f"False positives: {len(fp_df)}   False negatives: {len(fn_df)}")

In [ ]:
# ── FP / FN pair viewer ───────────────────────────────────────────────────────
import rasterio

def read_tile_rgb(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    return percentile_stretch(to_display_rgb(torch.from_numpy(arr)))

def show_pairs(subset, title, n=N_SHOW):
    samples = subset.sample(min(n, len(subset)), random_state=SEED)
    fig, axes = plt.subplots(len(samples), 2, figsize=(8, len(samples) * 3.5))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(title, fontsize=12)
    for row, (_, r) in enumerate(samples.iterrows()):
        ref_rgb = read_tile_rgb(r["image_path1"])
        qry_rgb = read_tile_rgb(r["image_path2"])
        axes[row, 0].imshow(ref_rgb)
        axes[row, 0].set_title(f"REF  site={r['site']}")
        axes[row, 0].axis("off")
        axes[row, 1].imshow(qry_rgb)
        axes[row, 1].set_title(
            f"QUERY  gt={r['str_labels']}  p={r['softmax_bleach_scores']:.2f}"
        )
        axes[row, 1].axis("off")
    fig.tight_layout()
    return fig

fig_fp = show_pairs(fp_df, "False Positives — predicted bleached, actually healthy")
fig_fp.savefig(os.path.join(FIGURES_DIR, "failure_fp.png"), dpi=150, bbox_inches="tight")
plt.show()

fig_fn = show_pairs(fn_df, "False Negatives — predicted healthy, actually bleached")
fig_fn.savefig(os.path.join(FIGURES_DIR, "failure_fn.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Score distributions: correct vs. incorrect ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for is_correct, grp in val_df.groupby("correct"):
    label = "correct" if is_correct else "incorrect"
    ax.hist(grp["softmax_bleach_scores"], bins=20, range=(0, 1),
            alpha=0.6, label=label)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Softmax bleach score")
ax.set_ylabel("Count")
ax.set_title("Score distribution — correct vs. incorrect predictions")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "failure_score_dist.png"), dpi=150)
plt.show()

In [ ]:
# ── Filename near-duplicate check ─────────────────────────────────────────────
# Flags if the same loc filename (e.g. loc001.tif) appears in both train and val.
# A match doesn't mean leakage (different dates/sites), but worth inspecting.
train_meta = dl_train.dataset.meta
val_meta   = dl_val.dataset.meta

shared_filenames = set(train_meta["filename"]) & set(val_meta["filename"])
print(f"Shared filenames across train/val: {len(shared_filenames)}")

if shared_filenames:
    overlap = val_meta[val_meta["filename"].isin(shared_filenames)]
    print(overlap[["site", "label", "date", "filename"]].to_string(index=False))